In [ ]:
## Install required dependencies: tiffile, monai, einops, torch, torchvision, numpy, matplotlib, cv2, scipy, skimage
import os
import time
import monai
import numpy as np
import pandas as pd
import einops
import tifffile
from torchsummary import summary

import matplotlib.pyplot as plt
from PIL import Image
import pickle
import glob
import shutil
import tempfile
import re
import io

from monai.losses import DiceCELoss, DiceLoss
from monai.inferers import sliding_window_inference

from monai.utils import first, set_determinism

from monai.config import print_config
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR, UNETR,FlexibleUNet, RegUNet
from monai.transforms import AsDiscrete
import networkx as nx
from monai.data import CacheDataset, DataLoader, Dataset, decollate_batch
import cv2 as cv
import torch
import torchvision
from torchvision import transforms

from collections import deque
import skimage
import scipy

Methods used for the connectivity analysis are defined in the cell under.

In [ ]:
def get_metrics(G):
    total_length = 0
    total_avg_thickness = 0
    total_dead_ends = 0
    total_connections = 0
    total_edges = 0
    total_nodes = 0
    tot_length_edge = 0
    total_connections = 0
    
    if 0 in G:
        G.remove_node(0)

    for node, data in G.nodes(data=True):
        total_dead_ends += data.get('dead_ends', 0)
        connected_nodes = list(G[node])  # get all nodes connected to the current node
        total_nodes += 1

        for cn in connected_nodes:
            edge_data = G[node][cn]

            # If edge_data contains direct attributes
            if 'length' in edge_data or 'shortest_path' in edge_data:
                if 'length' in edge_data:
                    total_length += edge_data['length']
                elif 'shortest_path' in edge_data:
                    total_length += edge_data['shortest_path']

                total_avg_thickness += edge_data['avg_thickness']
                total_edges += 1

            # If edge_data is nested
            else:
                for key in edge_data:
                    path_data = edge_data[key]

                    if 'length' in path_data:
                        total_length += path_data['length']
                    elif 'shortest_path' in path_data:
                        total_length += path_data['shortest_path']

                    total_avg_thickness += path_data['avg_thickness']
                    total_edges += 1

    # Compute average values for all graphs combined
    if total_edges > 0 and total_nodes > 0:
        average_length = total_length / total_edges
        average_avg_thickness = total_avg_thickness / total_edges
        avg_connections = total_edges / total_nodes
        avg_dead_ends = total_dead_ends / total_nodes
    else:
        avg_connections = total_edges
        average_length = total_length
        average_avg_thickness = total_avg_thickness
        avg_dead_ends = total_dead_ends
       
    metrics = [total_nodes, total_edges,avg_connections, total_dead_ends, avg_dead_ends, average_length, average_avg_thickness]
    return metrics
        
## Function used to find the nearest dendrite pixel given centroid coordinates of an osteocyte
# Centroid: x,y coordinates of osteocyte - tuple
# edge_labels: 512x512 image of all unique labels returned from connectomics analysis of dendrite mask - np.array
# edge: id of edge label - int
# return: nearest dendrite point - tuple, or none if none found
def get_nearest_dendrite_point(centroid, edge_labels, edge):
    rows, cols = edge_labels.shape
    max_radius = max(rows, cols)

    for r in range(max_radius):
        for i in range(-r, r+1):
            for j in range(-r, r+1):
                if (abs(i) == r or abs(j) == r):  # Check the perimeter of the square
                    new_y = centroid[1] + i
                    new_x = centroid[0] + j
                    if 0 <= new_x < cols and 0 <= new_y < rows and edge_labels[new_y][new_x] == edge:
                        return (new_y, new_x)
    return None  

# Check if a point is valid or not
# return : True or False
def is_valid(x, y, rows, cols, visited, edge_labels, edge):
    return (0 <= x < rows) and (0 <= y < cols) and (visited[x][y] == False) and (edge_labels[x][y] == edge)

## BFS implementation to calculate shortest path 
# edge_labels: 512x512 image of all unique labels returned from connectomics analysis of dendrite mask - np.array
# edge: id of edge label - int
# start: x,y coordinates of start point of dendrite - tuple
# end: x,y coordinates of end point of dendrite - tuple
# return: distance of shortest path, -1 if none is found -int
def shortest_path(edge_labels, edge, start, end):
    rows, cols = edge_labels.shape

    dx = [1, 0, -1, 0]  # Directions for moving in rows
    dy = [0, 1, 0, -1]  # Directions for moving in columns

    visited = [[False for _ in range(cols)] for _ in range(rows)]  # Visited matrix

    queue = deque()
    queue.append((start, 0))  # Start BFS from the 'start' node

    while queue:
        (y, x), dist = queue.popleft()  # Changed (x, y) to (y, x)

        # If this point is the end point, return the distance
        if (y, x) == end:  
            return dist

        for direction in range(4):  # Check all four possible directions
            new_y, new_x = y + dy[direction], x + dx[direction]  # Changed order

            if is_valid(new_y, new_x, rows, cols, visited, edge_labels, edge):  # Swapped order of new_x and new_y
                visited[new_y][new_x] = True
                queue.append(((new_y, new_x), dist + 1))

    # If no path found, return -1
    return -1

## Function to calculate amount of dead ends 
# y: array of y values for a dendrite label segment - np.array
# x: array of y values for a dendrite label segment - np.array
# edge_labels: 512x512 image of all unique labels returned from connectomics analysis of dendrite mask - np.array
# return number of dead ends - int
def count_dead_ends(y,x, edge_labels):
    """
    Count the number of dead ends in a given edge component.
    """
    dead_ends = 0

    directions = [(1, 0), (0, 1), (-1, 0), (0, -1), (1, 1), (-1, -1), (1, -1), (-1, 1)]

    for i in range(len(x)):
        count = 0
        for dx, dy in directions:
            new_x, new_y = x[i] + dx, y[i] + dy

            if 0 <= new_x < edge_labels.shape[1] and 0 <= new_y < edge_labels.shape[0] and edge_labels[new_y, new_x] >0:
                count += 1
        if count == 1: 
            dead_ends += 1
    return dead_ends

## function to provide nicer interface to visualise graph
# G: graph object to visualise - NetworkX Graph
# avg_thickness: average thickness of whole graph, can be removed: int
def visualize_graph(G, avg_thickness):
    # Node positions and colors
    if 0 in G:
        G.remove_node(0)

    #pos = {node: data['centroid'] for (node, data) in G.nodes(data=True)}
    pos = {node: (data['centroid'][0], -data['centroid'][1]) for (node, data) in G.nodes(data=True)}

    print("Average canaliculi thickness of the image is: {:.2f} micrometer".format(avg_thickness*75/512)) ## 75/512 responds to spatial resolution / pixel resolution ratio
    print("The LCN has a degree of:", G.degree)
    print("_______________________________________________________")
    # Iterate through nodes and print node & edge details in the desired format
    for node, data in G.nodes(data=True):
        #if not data.get('isFake', False):
        connected_nodes = list(G[node])  # get all nodes connected to current node

        total_connections = sum([len(G[node][cn]) for cn in connected_nodes])
        print(f"Node {node} has a total of {total_connections} connections to {len(connected_nodes)} different nodes, Node data: {data}")
        for cn in connected_nodes:
            edge_data = G[node][cn]
            print(f"- Connection with node {cn}, {edge_data}")
        print("_______________________________________________________")

    # Visualize the graph
    nx.draw(G, pos, with_labels=True)
    plt.show()
        
## function to calculate connectomics analysis given a segmentation of 512x512 where pixel values 0=backround, 1 = osteocytes, 2=dendrites
def connectomics_analysis(segmentation):
       
    pixel_ratio = 75/512 ## 75 micrometer / 512 pixel spatial resolution / pixel resolution
    
    segmentation_image= segmentation.astype(np.int8)

    osteocytes = np.zeros_like(segmentation_image)
    dendrites = np.zeros_like(segmentation_image)

    osteocytes[segmentation_image == 1] = 1
    dendrites[segmentation_image == 2] = 2
    nr_dendrite_pixels = np.sum(dendrites==2) ## total count of dendrite pixels 
    dendrite_skeleton = skimage.morphology.skeletonize(dendrites==2) ## skeletonise dendrite label
    average_LCN_thickness = nr_dendrite_pixels / np.sum(dendrite_skeleton)
    average_LCN_thickness = average_LCN_thickness * pixel_ratio
    osteocytes = osteocytes.astype(np.uint8)
    # Dilation of osteocytes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    dilated_osteocytes = cv2.dilate(osteocytes, kernel, iterations=1)

    # Connected component labeling
    _, labels = cv2.connectedComponents(dilated_osteocytes)

    # Graph construction
    G = nx.MultiGraph()
    # Node extraction
    nodes = set(labels[dilated_osteocytes == 1])  # get all unique labels as nodes
    for node in nodes:
        y, x = (labels == node).nonzero()  # extract x,y values for each node
        centroid = (int(x.mean()), int(y.mean()))  # Compute centroid 
        G.add_node(node, centroid=centroid, dead_ends=0)  # 

    # Edge extraction using morphological operations
    nr_dendrite_pixels = np.sum(dendrites==2)
    dendrites = dendrites.astype(np.uint8)
    # Apply morphological operations to enhance connectivity
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    dendrites = cv2.dilate(dendrites, kernel, iterations = 1)
    
    # perform similar extraction of edges 
    edge_labels, nr_dendrites   = scipy.ndimage.label(dendrites)

    # edge extraction
    edges = set(np.unique(edge_labels))  # get all unique labels as edges
    edges.remove(0) # scipy.ndimage.label always return 0 value as a unique edge, so we remove it
    for edge in edges:
        y, x = (edge_labels == edge).nonzero()  # extract x,y values for each edge
        nodes_in_edge = set(labels[y, x]) # find osteocytes that overlap the current dendrite structure
        
        if(0 in nodes_in_edge):
            nodes_in_edge.remove(0)

        skel_edge = skimage.morphology.skeletonize(edge_labels == edge) ## skeleton of dendrite structure

        if len(nodes_in_edge) == 1: # only goes through one node 
           
            sy,sx = (skel_edge ==1).nonzero()
            a = np.zeros((512,512))
            a[sy,sx] = 30
            a = a + labels
            dead_ends = count_dead_ends(sy,sx, a) 

            G.nodes[list(nodes_in_edge)[-1]]['dead_ends'] += dead_ends 

        # If dendrite component is connected to several osteocytes in the segmentation
        elif (len(nodes_in_edge) > 2):  ## Have to calculate shortest path
            processed_pairs = set()  # To keep track of the node pairs we've already processed
            for node in nodes_in_edge:  
                start_point = get_nearest_dendrite_point(G.nodes[node]['centroid'], edge_labels, edge) # get nearest dendrite point so we can compute shortest path between the two osteocytes
                for node2 in nodes_in_edge:    
                    if node != node2 and node in G.nodes and node2 in G.nodes:
                        pair = tuple(sorted([node, node2]))  # Sort the node pair so (node, node2) is the same as (node2, node)
                        if pair not in processed_pairs:  # Check if the pair hasn't been processed
                            processed_pairs.add(pair)  # Add the pair to our set
                            end_point = get_nearest_dendrite_point(G.nodes[node2]['centroid'], edge_labels, edge)
                            distance = shortest_path(edge_labels, edge, start_point, end_point)
                            if distance != -1:
                                original_edge = (edge_labels == edge)
                                original_edge = original_edge.astype(np.uint8)
                                original_edge = cv2.erode(original_edge, kernel, iterations=1)
                                avg_thickness = np.sum(original_edge) / np.sum(skel_edge)
                                avg_thickness = avg_thickness * pixel_ratio ## get average thickness of the connection
                                G.add_edge(node, node2, shortest_path=(distance*pixel_ratio), avg_thickness=avg_thickness) # create edge object in our graph

        # if dendrite component is connected to only 2 osteocytes in segmentation                     
        elif(len(nodes_in_edge)==2):
            node1, node2 = nodes_in_edge
            if node1 != node2 and node1 in G.nodes and node2 in G.nodes:

                start_point = get_nearest_dendrite_point(G.nodes[node1]['centroid'], edge_labels, edge)
                end_point = get_nearest_dendrite_point(G.nodes[node2]['centroid'], edge_labels, edge)
                distance = shortest_path(edge_labels, edge, start_point, end_point)
                if distance != -1:

                    original_edge = (edge_labels == edge)
                    original_edge = original_edge.astype(np.uint8)
                    original_edge = cv2.erode(original_edge, kernel, iterations = 1)
                    avg_thickness = np.sum(original_edge) / np.sum(skel_edge)
                    avg_thickness = avg_thickness * pixel_ratio
                    G.add_edge(node1, node2, length = (distance * pixel_ratio), avg_thickness = avg_thickness)
                    sy,sx = (skel_edge ==1).nonzero()
                    a = np.zeros((512,512))
                    a[sy,sx] = 1 # Can be any value expect 0 (background)
                    a = a + labels # Add osteocyte labels in order to not count dendrites bordering osteocytes as  dead ends 
                    dead_ends = count_dead_ends(sy,sx, a) 
                    G.nodes[node1]['dead_ends'] += dead_ends 
                    G.nodes[node2]['dead_ends'] += dead_ends 

    return G, average_LCN_thickness

def compute_averages(graph_list):
    total_length = 0
    total_avg_thickness = 0
    total_dead_ends = 0
    total_edges = 0
    total_nodes = 0

    #tot_length_edge = 0
    #total_connections = 0
    
    for G in graph_list:
        graph_count = len(graph_list)
        
        if 0 in G:
            G.remove_node(0)

        for node, data in G.nodes(data=True):
            total_dead_ends += data.get('dead_ends', 0)
            connected_nodes = list(G[node])  # get all nodes connected to the current node
            total_nodes += 1

            for cn in connected_nodes:
                edge_data = G[node][cn]

                # If edge_data contains direct attributes
                if 'length' in edge_data or 'shortest_path' in edge_data:
                    if 'length' in edge_data:
                        total_length += edge_data['length']
                    elif 'shortest_path' in edge_data:
                        total_length += edge_data['shortest_path']

                    total_avg_thickness += edge_data['avg_thickness']
                    total_edges += 1

                # If edge_data is nested
                else:
                    for key in edge_data:
                        path_data = edge_data[key]

                        if 'length' in path_data:
                            total_length += path_data['length']
                        elif 'shortest_path' in path_data:
                            total_length += path_data['shortest_path']

                        total_avg_thickness += path_data['avg_thickness']
                        total_edges += 1

    # Compute average values for all graphs combined
    if total_edges > 0 and total_nodes > 0:
        average_length = total_length / total_edges
        average_avg_thickness = total_avg_thickness / total_edges
        avg_connections = total_edges / total_nodes
        avg_dead_ends = total_dead_ends / total_nodes
        average_nodes = total_nodes / graph_count    
    else:
        avg_connections = 0
        average_length = 0
        average_avg_thickness = 0
        
    avg_connections = total_edges / total_nodes
    print(f"Average nodes(for all graphs): {average_nodes}")
    print(f"Average Dead Ends per node: {avg_dead_ends}")
    print(f"Average Connections per node: {avg_connections}")
    print(f"Average Length of each connection(for all graphs): {average_length}")
    print(f"Average Avg Thickness of connections(for all graphs): {average_avg_thickness}")




Given your custom dataset, you can run the code under to perform inference using the pre-trained Attention UNet model. The model output will be used for the connectivity analysis, and a final dataframe will be written with the calulated results.

In [ ]:
class CustomTiffDataset(Dataset):
    def __init__(self, transform=None):
        
        self.transform = transform

        self.file_list = sorted(glob.glob(os.path.join( r"data/", "*.tif")))

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        image_path = self.file_list[idx]
        image = Image.open(image_path)
        image = transforms.functional.pil_to_tensor(image)
        image = image.float()
        
        if self.transform:
            image = self.transform(image)

        return image

# Define data transformations (you can customize these)
data_transform = transforms.Compose([
    transforms.Normalize(0.5, 0.2)
])

# Create an instance of your custom dataset
dataset = CustomTiffDataset(transform=data_transform)

# Create a data loader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
post_pred = AsDiscrete(argmax=True, to_onehot=3)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

####
## Inference
####
spatial_dims = 2
in_channels = 1       # Grayscale images have 1 channel
out_channels = 3      # You want to classify pixels into 3 classes
channels = (16, 32, 64, 128, 256)  # Number of channels at each depth level. You can modify this based on your requirement.
strides = (2, 2, 2, 2)  # Strides for each depth level. You can modify this based on your requirement.

# Initialize the network
model = monai.networks.nets.AttentionUnet(
    spatial_dims=spatial_dims,
    in_channels=in_channels,
    out_channels=out_channels,
    channels=channels,
    strides=strides
)

## Load pre-trained network
model.load_state_dict(torch.load(r"attentionunet_lowlr.pth", map_location=torch.device('cpu')))
model.to(device)
model.eval()
all_metrics = []
graph_list = []
avg_big = 0

with torch.no_grad():
    for batch in dataloader:
        batch = batch[:,0,:,:]
        batch = torch.unsqueeze(batch, 1)

        outputs = model(batch)
        outputs = torch.argmax(torch.softmax(outputs, dim = 1), dim=1)
        outputs = outputs.numpy()
        
        for i in range(outputs.shape[0]):
            G, avg = connectomics_analysis(outputs[i])
            all_metrics.append(get_metrics(G))
            avg_big += avg
            graph_list.append(G)
            #visualize_graph(G, avg)
            
    #compute_averages(graph_list)
    #print(avg_big/len(graph_list))


columns = ['nr_of_nodes', 'nr_of_edges', 'avg_connections_per_node', 'total_dead_ends', 'avg_dead_ends_per_node', 'average_length_per_connection', 'average_thickness_per_connection']
result_df = pd.DataFrame(columns=columns)
     
for metric in all_metrics:
    result_df = pd.concat([result_df, pd.DataFrame([metric], columns=columns)], ignore_index=True)

# Export the DataFrame to a spreadsheet (e.g., CSV file)
result_df.to_csv('connectivity_analysis_results.csv', index=False)